In [ ]:
"/content/drive/MyDrive/PCB_MC/YOLO/components_only"

# Cross validation organization and Preparation for all the subset



1. Components Only
2. Full Dataset
3. Missing Only
4. Non missing






In [ ]:
/content/drive/MyDrive/PCB_MC/Data/full_dataset
/content/drive/MyDrive/PCB_MC/Data/missing_only

# Task
Create a 5-fold cross-validation dataset structure from the existing image and label data located in the `train`and `valid` subdirectories within `/content/drive/MyDrive/PCB_MC/YOLO/components_only`. The process should consolidate all data, split it into 5 folds, where each fold has its own `train` and  `valid` sets, and then clean up any temporary directories.

## Prepare Environment and Paths

### Subtask:
Set up the necessary Python libraries and define file paths.


**Reasoning**:
The user wants to import necessary libraries and define file paths as part of environment setup. This will be done in a single code block.



In [1]:
import os
import shutil
from sklearn.model_selection import KFold, train_test_split

base_data_path = '/content/drive/MyDrive/PCB_MC/Data/full_dataset'
kfold_output_path = '/content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data'

print(f"Base data path set to: {base_data_path}")
print(f"K-fold output path set to: {kfold_output_path}")

Base data path set to: /content/drive/MyDrive/PCB_MC/Data/full_dataset
K-fold output path set to: /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data


## Consolidate All Data

### Subtask:
Create temporary `all_data/images` and `all_data/labels` directories. Copy all images and their corresponding labels from the existing `train` and `valid`  subfolders within `/content/drive/MyDrive/PCB_MC/YOLO/components_only` into these temporary 'all_data' directories. This step ensures all your data is in one place before the K-fold split.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Reasoning**:
To consolidate all images and labels into a single directory, I will create the temporary 'all_data' structure, clear its contents if it exists, and then iterate through the 'train' and 'valid' folders to copy all relevant files. This ensures all data is in one place before K-fold splitting.



In [2]:
all_data_path = '/content/all_data'
all_images_path = os.path.join(all_data_path, 'images')
all_labels_path = os.path.join(all_data_path, 'labels')

# Create/clean all_data directories
for path in [all_images_path, all_labels_path]:
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path)
print(f"Created/cleaned temporary directories: {all_images_path} and {all_labels_path}")

total_images_copied = 0
total_labels_copied = 0

data_subsets = ['train', 'valid', 'test']

for subset in data_subsets:
    subset_images_path = os.path.join(base_data_path, subset, 'images')
    subset_labels_path = os.path.join(base_data_path, subset, 'labels')

    # Copy images
    if os.path.exists(subset_images_path):
        for filename in os.listdir(subset_images_path):
            if filename.endswith(('.jpg', '.jpeg', '.png')):
                shutil.copy(os.path.join(subset_images_path, filename), all_images_path)
                total_images_copied += 1

    # Copy labels
    if os.path.exists(subset_labels_path):
        for filename in os.listdir(subset_labels_path):
            if filename.endswith('.txt'):
                shutil.copy(os.path.join(subset_labels_path, filename), all_labels_path)
                total_labels_copied += 1

print(f"\nConsolidation complete: ")
print(f"Total images copied to {all_images_path}: {total_images_copied}")
print(f"Total labels copied to {all_labels_path}: {total_labels_copied}")

Created/cleaned temporary directories: /content/all_data/images and /content/all_data/labels

Consolidation complete: 
Total images copied to /content/all_data/images: 615
Total labels copied to /content/all_data/labels: 615


**Reasoning**:
Now that all data is consolidated, the next step is to get a list of all image paths, which will be used as the basis for the K-fold split. The image paths are used to ensure the labels are correctly matched during splitting.



In [3]:
all_image_files = sorted([os.path.join(all_images_path, f) for f in os.listdir(all_images_path) if f.endswith(('.jpg', '.jpeg', '.png'))])

print(f"Found {len(all_image_files)} image files for K-fold splitting.")
# print first 5 files to verify
print(f"First 5 image files: {all_image_files[:5]}")

Found 615 image files for K-fold splitting.
First 5 image files: ['/content/all_data/images/ACM-109_Bottom_jpg.rf.2b36e9e46bb6541f21b0349036893a23.jpg', '/content/all_data/images/ACM-109_Bottom_jpg.rf.42ea23c6d4608af7d16e5f96cc97d61a.jpg', '/content/all_data/images/ACM-109_Bottom_jpg.rf.803655de95a892d2eb8964823288c192.jpg', '/content/all_data/images/ACM-109_Bottom_jpg.rf.8c2aa6441823c330abbeed95423e9917.jpg', '/content/all_data/images/ACM-109_Top_jpg.rf.bd0e85a6ed49534eb1dc40aef62d77e2.jpg']


## Perform K-fold Split and Create Fold Directories

### Subtask:
Split the consolidated image data into 5 folds. For each fold, create `train` and `valid` splits, and set up the corresponding directory structure in the `kfold_data` path.

#### Instructions
1. Initialize `KFold` with `n_splits=5`, `shuffle=True`, and a `random_state` (e.g., 42) for reproducibility.
2. Create the base `kfold_output_path` directory if it does not already exist.
3. Iterate through the splits generated by `KFold`. For each `fold_idx`:
    *   Create a directory for the current fold (e.g., `kfold_data/fold_0`).
    *   Take the indices for the current fold's training and testing sets from `KFold`.
    *   Split the training set further into actual training and validation sets (e.g., an 80/20 split of the total data would mean the initial 'train' from KFold is split into 80% train and 20% valid).
    *   For each of `train` and `valid` in the current fold:
        *   Create `images` and `labels` subdirectories (e.g., `kfold_data/fold_0/train/images`).
        *   Copy the corresponding image and label files from the `all_data` consolidated directories into these new fold-specific subdirectories. Ensure that for each image, its corresponding label file is also copied.

## Perform K-fold Split and Create Fold Directories

### Subtask:
Split the consolidated image data into 5 folds. For each fold, create `train` and `valid` splits, and set up the corresponding directory structure in the `kfold_data` path.

#### Instructions
1. Initialize `KFold` with `n_splits=5`, `shuffle=True`, and a `random_state` (e.g., 42) for reproducibility.
2. Create the base `kfold_output_path` directory if it does not already exist.
3. Iterate through the splits generated by `KFold`. For each `fold_idx`:
    *   Create a directory for the current fold (e.g., `kfold_data/fold_0`).
    *   Take the indices for the current fold's training and testing sets from `KFold`.
    *   Split the training set further into actual training and validation sets (e.g., an 80/20 split of the total data would mean the initial 'train' from KFold is split into 80% train and 20% valid.
    *   For each of `train`, `valid`, and `test` in the current fold:
        *   Create `images` and `labels` subdirectories (e.g., `kfold_data/fold_0/train/images`).
        *   Copy the corresponding image and label files from the `all_data` consolidated directories into these new fold-specific subdirectories. Ensure that for each image, its corresponding label file is also copied.

In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Create base K-fold output directory if it doesn't exist
if os.path.exists(kfold_output_path):
    shutil.rmtree(kfold_output_path)
os.makedirs(kfold_output_path, exist_ok=True)
print(f"Created base K-fold output directory: {kfold_output_path}")

def copy_files(file_list, destination_images_path, destination_labels_path, source_all_labels_path):
    for img_path in file_list:
        # Ensure destination directories exist
        os.makedirs(destination_images_path, exist_ok=True)
        os.makedirs(destination_labels_path, exist_ok=True)

        # Copy image
        shutil.copy(img_path, destination_images_path)

        # Copy corresponding label
        img_filename = os.path.basename(img_path)
        label_filename = os.path.splitext(img_filename)[0] + '.txt'
        src_label_path = os.path.join(source_all_labels_path, label_filename)
        if os.path.exists(src_label_path):
            shutil.copy(src_label_path, destination_labels_path)
        else:
            # This case indicates an image without a label, which shouldn't happen for YOLO datasets
            print(f"Warning: Label file not found for image: {img_filename} at {src_label_path}")


for fold_idx, (train_indices, valid_indices) in enumerate(kf.split(all_image_files)):
    print(f"\nProcessing Fold {fold_idx}")

    fold_path = os.path.join(kfold_output_path, f'fold_{fold_idx}')
    os.makedirs(fold_path, exist_ok=True)

    # KFold's train_indices represent ~80% of the data.
    # KFold's valid_indices (formerly test) represent ~20% of the data.
    # We want 80% train, 20% valid, with no separate test set at this stage.

    fold_train_files = [all_image_files[i] for i in train_indices]
    fold_valid_files = [all_image_files[i] for i in valid_indices]

    print(f"  Train set size: {len(fold_train_files)} images")
    print(f"  Valid set size: {len(fold_valid_files)} images")


    # Define and create subdirectories for this fold
    train_images_path = os.path.join(fold_path, 'train', 'images')
    train_labels_path = os.path.join(fold_path, 'train', 'labels')
    valid_images_path = os.path.join(fold_path, 'valid', 'images')
    valid_labels_path = os.path.join(fold_path, 'valid', 'labels')


    # Copy files for each split
    print("  Copying train files...")
    copy_files(fold_train_files, train_images_path, train_labels_path, all_labels_path)
    print("  Copying valid files...")
    copy_files(fold_valid_files, valid_images_path, valid_labels_path, all_labels_path)

print("\nK-fold data preparation complete.")

Created base K-fold output directory: /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data

Processing Fold 0
  Train set size: 492 images
  Valid set size: 123 images
  Copying train files...
  Copying valid files...

Processing Fold 1
  Train set size: 492 images
  Valid set size: 123 images
  Copying train files...
  Copying valid files...

Processing Fold 2
  Train set size: 492 images
  Valid set size: 123 images
  Copying train files...
  Copying valid files...

Processing Fold 3
  Train set size: 492 images
  Valid set size: 123 images
  Copying train files...
  Copying valid files...

Processing Fold 4
  Train set size: 492 images
  Valid set size: 123 images
  Copying train files...
  Copying valid files...

K-fold data preparation complete.


In [5]:
print(f"Cleaning up temporary directory: {all_data_path}")
if os.path.exists(all_data_path):
    shutil.rmtree(all_data_path)
print("Temporary 'all_data' directory cleaned up.")

Cleaning up temporary directory: /content/all_data
Temporary 'all_data' directory cleaned up.
